In [1]:
import pandas as pd
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")

In [2]:
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

In [3]:
train_df = train_df.sort_values(by="timestamp_num").reset_index(drop=True)
valid_df = valid_df.sort_values(by="timestamp_num").reset_index(drop=True)
test_df = test_df.sort_values(by="timestamp_num").reset_index(drop=True)

In [4]:
train_df = train_df.sort_values(by=['timestamp_num'])
valid_df = valid_df.sort_values(by=['timestamp_num'])
test_df = test_df.sort_values(by=['timestamp_num'])

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [5]:
#Base learner: train trên tập train, early stopping trên tập valid. Meta learner: train trên tập valid (tập meta validation), đánh giá trên tập test
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, GRU, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Kiểm tra GPU
print(f"TensorFlow phiên bản: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"Số lượng GPU khả dụng cho TensorFlow: {len(gpus)}")
if gpus:
    for gpu in gpus:
        print("Trạng thái GPU:", gpu)

# ---------------------------------------------------------
# 1. TÁCH DỮ LIỆU & ĐỊNH DẠNG ĐẦU VÀO
# ---------------------------------------------------------
print("Chuẩn bị dữ liệu...")
# Lấy X, y từ các tập dữ liệu gốc
X_train = train_df.drop(columns=['label']).values
y_train = train_df['label'].values

X_valid = valid_df.drop(columns=['label']).values
y_valid = valid_df['label'].values

X_test = test_df.drop(columns=['label']).values
y_test = test_df['label'].values

num_classes = len(np.unique(y_train))
print(f"Số lượng classes: {num_classes}")

# Reshape X sang dạng 3D (samples, time_steps=1, features) cho LSTM/GRU
X_train_3d = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_valid_3d = X_valid.reshape(X_valid.shape[0], 1, X_valid.shape[1])
X_test_3d = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])

# ---------------------------------------------------------
# 2. XÂY DỰNG VÀ HUẤN LUYỆN CÁC MÔ HÌNH CƠ SỞ (TẦNG 1)
# ---------------------------------------------------------
print("\n--- Bắt đầu huấn luyện Tầng 1 trên TẬP TRAIN, Early Stopping trên TẬP VALID ---")

# 1. Gradient Boosting (Scikit-learn không hỗ trợ eval_set bên ngoài cho Early Stopping)
print("- Đang huấn luyện GBM (HistGradientBoosting trên CPU)...")
gbm = HistGradientBoostingClassifier()
gbm.fit(X_train, y_train)

# 2. LightGBM (Cập nhật Early Stopping)
print("- Đang huấn luyện LightGBM (trên CPU đa luồng)...")
lgbm = LGBMClassifier(verbose=-1, n_jobs=-1)
lgbm.fit(
    X_train, y_train, 
    eval_set=[(X_valid, y_valid)], 
    callbacks=[lgb.early_stopping(stopping_rounds=10, verbose=False)]
)

# 3. Bagging (Giữ nguyên, bản chất là ensemble độc lập, không có Early Stopping)
print("- Đang huấn luyện Bagging (1000 cây trên CPU)... (Có thể mất vài phút)")
dt_base = DecisionTreeClassifier(max_depth=15)
bagging = BaggingClassifier(estimator=dt_base, n_estimators=1000, max_samples=0.01, n_jobs=2)
bagging.fit(X_train, y_train)

# 4. XGBoost (Cập nhật Early Stopping vào constructor)
print("- Đang huấn luyện XGBoost (trên GPU)...")
xgb = XGBClassifier(
    use_label_encoder=False, 
    eval_metric='mlogloss', 
    tree_method='hist', 
    device='cuda',
    early_stopping_rounds=10
)
xgb.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=0)

# 5. CatBoost (Cập nhật Early Stopping)
print("- Đang huấn luyện CatBoost (trên GPU)...")
cat = CatBoostClassifier(
    verbose=0, 
    task_type='GPU',
    early_stopping_rounds=10
)
cat.fit(X_train, y_train, eval_set=(X_valid, y_valid))

# ---------------------------------------------------------
# Khối Deep Learning (LSTM & GRU)
# Không còn tách 10% từ X_train nữa, sử dụng thẳng X_valid_3d
# ---------------------------------------------------------
nn_es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train LSTM
print("- Đang huấn luyện LSTM (trên GPU)...")
lstm_model = Sequential([
    LSTM(128, input_shape=(1, X_train.shape[1])),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

lstm_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
lstm_model.fit(
    X_train_3d, y_train, 
    validation_data=(X_valid_3d, y_valid), 
    epochs=30, batch_size=1024, 
    callbacks=[nn_es], verbose=1
)

# Train GRU
print("- Đang huấn luyện GRU (trên GPU)...")
gru_model = Sequential([
    GRU(128, input_shape=(1, X_train.shape[1])),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

gru_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
gru_model.fit(
    X_train_3d, y_train, 
    validation_data=(X_valid_3d, y_valid), 
    epochs=20, batch_size=1024, 
    callbacks=[nn_es], verbose=1
)


# ---------------------------------------------------------
# 3. TẠO CÁC ĐẶC TRƯNG META TỪ TẬP VALID (HOLD-OUT) VÀ TEST
# ---------------------------------------------------------
print("\n--- Trích xuất Meta-Features từ Tầng 1 ---")
def get_meta_features(X_2d, X_3d):
    p_gbm = gbm.predict_proba(X_2d)
    p_lgbm = lgbm.predict_proba(X_2d)
    p_bag = bagging.predict_proba(X_2d)
    p_xgb = xgb.predict_proba(X_2d)
    p_cat = cat.predict_proba(X_2d)
    p_lstm = lstm_model.predict(X_3d, verbose=0)
    p_gru = gru_model.predict(X_3d, verbose=0)
    
    return np.hstack([p_gbm, p_lgbm, p_bag, p_xgb, p_cat, p_lstm, p_gru])

print("- Tạo dữ liệu huấn luyện Meta-Model trên tập VALID (Hold-out)...")
X_meta_train = get_meta_features(X_valid, X_valid_3d)

print("- Tạo dữ liệu đánh giá Meta-Model trên tập TEST...")
X_meta_test = get_meta_features(X_test, X_test_3d)


# ---------------------------------------------------------
# 4. HUẤN LUYỆN META-MODEL (TẦNG 2) VÀ ĐÁNH GIÁ TRÊN TEST
# ---------------------------------------------------------
print("\n--- Huấn luyện Meta-Model (EnsembleGuard Neural Network) Tầng 2 ---")
meta_model_nn = Sequential([
    Dense(64, activation='relu', input_shape=(X_meta_train.shape[1],)),
    Dense(num_classes, activation='softmax')
])

meta_model_nn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# Vẫn giữ nguyên việc huấn luyện trên tập meta validation (X_meta_train, y_valid)
meta_model_nn.fit(X_meta_train, y_valid, epochs=20, batch_size=1024, verbose=1)


print("\n==============================================")
print("ĐÁNH GIÁ ENSEMBLE TRÊN TẬP TEST ĐỘC LẬP TỪ META-MODEL")
print("==============================================")

meta_preds_proba = meta_model_nn.predict(X_meta_test)
meta_preds_classes = np.argmax(meta_preds_proba, axis=1)

print("\n--- Báo cáo Phân loại (Classification Report) ---")
print(classification_report(y_test, meta_preds_classes, digits=4))

from sklearn.metrics import roc_auc_score

acc = accuracy_score(y_test, meta_preds_classes)
prec = precision_score(y_test, meta_preds_classes, average='macro', zero_division=0)
rec = recall_score(y_test, meta_preds_classes, average='macro', zero_division=0)
f1 = f1_score(y_test, meta_preds_classes, average='macro', zero_division=0)

if num_classes == 2:
    auc_roc = roc_auc_score(y_test, meta_preds_proba[:, 1])
else:
    auc_roc = roc_auc_score(y_test, meta_preds_proba, multi_class='ovr', average='macro')

print(f"\n=> Tỷ lệ chính xác (Accuracy): {acc:.4f}")
print(f"=> Macro Precision:          {prec:.4f}")
print(f"=> Macro Recall:             {rec:.4f}")
print(f"=> Macro F1-Score:           {f1:.4f}")
print(f"=> AUC-ROC:                  {auc_roc:.4f}")

TensorFlow phiên bản: 2.19.0
Số lượng GPU khả dụng cho TensorFlow: 0
Chuẩn bị dữ liệu...
Số lượng classes: 11

--- Bắt đầu huấn luyện Tầng 1 trên TẬP TRAIN, Early Stopping trên TẬP VALID ---
- Đang huấn luyện GBM (HistGradientBoosting trên CPU)...
- Đang huấn luyện LightGBM (trên CPU đa luồng)...
- Đang huấn luyện Bagging (1000 cây trên CPU)... (Có thể mất vài phút)
- Đang huấn luyện XGBoost (trên GPU)...


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\callback.py:385: UserWarning: [15:29:14] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


- Đang huấn luyện CatBoost (trên GPU)...
- Đang huấn luyện LSTM (trên GPU)...


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/30
2413/2413 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - loss: 0.2416 - val_loss: 0.2190
Epoch 2/30
2413/2413 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - loss: 0.1138 - val_loss: 0.2122
Epoch 3/30
2413/2413 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - loss: 0.1074 - val_loss: 0.2191
Epoch 4/30
2413/2413 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - loss: 0.1040 - val_loss: 0.2259
Epoch 5/30
2413/2413 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - loss: 0.1009 - val_loss: 0.2216
- Đang huấn luyện GRU (trên GPU)...
Epoch 1/20
2413/2413 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - loss: 0.2477 - val_loss: 0.2318
Epoch 2/20
2413/2413 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 0.1142 - val_loss: 0.2400
Epoch 3/20
2413/2413 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 0.1084 - val_loss: 0.2377
Epoch 4/20
2413/2413 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 0.1045 - val_loss: 0.2226
Epoch 5/20
2413/2413 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 0.1026 - val_loss: 0.2500
Epoch 6/20
2413/2413 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 0.1012 - va

...

- Tạo dữ liệu đánh giá Meta-Model trên tập TEST...


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



--- Huấn luyện Meta-Model (EnsembleGuard Neural Network) Tầng 2 ---
Epoch 1/20


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


557/557 ━━━━━━━━━━━━━━━━━━━━ 1s 884us/step - accuracy: 0.8615 - loss: 0.6594
Epoch 2/20
557/557 ━━━━━━━━━━━━━━━━━━━━ 1s 867us/step - accuracy: 0.9776 - loss: 0.0609
Epoch 3/20
557/557 ━━━━━━━━━━━━━━━━━━━━ 1s 886us/step - accuracy: 0.9790 - loss: 0.0564
Epoch 4/20
557/557 ━━━━━━━━━━━━━━━━━━━━ 0s 844us/step - accuracy: 0.9792 - loss: 0.0544
Epoch 5/20
557/557 ━━━━━━━━━━━━━━━━━━━━ 1s 871us/step - accuracy: 0.9800 - loss: 0.0528
Epoch 6/20
557/557 ━━━━━━━━━━━━━━━━━━━━ 1s 847us/step - accuracy: 0.9798 - loss: 0.0521
Epoch 7/20
557/557 ━━━━━━━━━━━━━━━━━━━━ 0s 828us/step - accuracy: 0.9803 - loss: 0.0510
Epoch 8/20
557/557 ━━━━━━━━━━━━━━━━━━━━ 0s 837us/step - accuracy: 0.9808 - loss: 0.0498
Epoch 9/20
557/557 ━━━━━━━━━━━━━━━━━━━━ 1s 874us/step - accuracy: 0.9810 - loss: 0.0491
Epoch 10/20
557/557 ━━━━━━━━━━━━━━━━━━━━ 0s 832us/step - accuracy: 0.9811 - loss: 0.0486
Epoch 11/20
557/557 ━━━━━━━━━━━━━━━━━━━━ 0s 830us/step - accuracy: 0.9811 - loss: 0.0482
Epoch 12/20
557/557 ━━━━━━━━━━━━━━━━━━━━ 